# Detector de colonias que funcione en cualquier laboratorio

Cuaderno para Google Colab con GPU. Entrena un detector con el dataset publico
AGAR, opcionalmente reforzado con placas sinteticas de condiciones de captura
variadas, y lo evalua sobre fotografias reales tomadas en dos montajes distintos.

## El problema que se quiere resolver

El sistema esta pensado para laboratorios sin presupuesto, que fotografian sus
placas con el telefono que tengan. El obstaculo no es contar colonias en una
fotografia concreta, eso ya funciona, sino que **el sistema siga funcionando
cuando cambia el laboratorio**.

Esa dificultad esta medida. Tres parametros calibrados en un montaje fallan en
otro:

| Parametro | Calibrado en | Falla en |
|-----------|--------------|----------|
| Estimador de densidad | Placas dobles | Contraluz, estima 10 en una placa de 352 |
| Filtro de color | Primer lote | Segundo lote, el tono del agar pasa de 43 a 56 |
| Umbral de deteccion | Contraluz | Placas dobles, sesgo de mas 8.5 colonias |

El pipeline actual, con CellSAM y preprocesamiento, alcanza un MAE de 2.83
colonias por placa en el laboratorio donde se ajusto, y se degrada fuera de el.

## La hipotesis de este cuaderno

Un detector que haya visto durante el entrenamiento muchas condiciones de
captura distintas no necesitaria recalibrarse en cada laboratorio. Se pone a
prueba entrenando con imagenes variadas y evaluando en dos montajes reales
**sin ajustar ningun parametro entre ellos**.

## Antes de empezar

Activa la GPU en Entorno de ejecucion, Cambiar tipo de entorno de ejecucion.

Sube a la raiz de tu Google Drive:

  `paquete_colab.zip`, con las fotografias reales, el conteo manual y los scripts

  `sinteticas_variadas.zip`, con las 800 placas sinteticas, opcional

  el dataset **AGAR completo**, que se consigue rellenando el formulario en
  https://agar.neurosys.com/ . Sin el, este cuaderno funciona con la muestra de
  40 imagenes, pero el resultado no sera concluyente.

## 1. Entorno

In [ ]:
import torch
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('SIN GPU. Activala antes de continuar.')

In [ ]:
!pip install --quiet ultralytics
from ultralytics import YOLO
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
import zipfile, shutil, subprocess, csv
from pathlib import Path
import pandas as pd
from PIL import Image

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
BASE = Path('/content/trabajo')
BASE.mkdir(exist_ok=True)

def descomprimir(nombre, destino):
    z = DRIVE / nombre
    if not z.exists():
        print(f'  {nombre}: no encontrado, se omite')
        return False
    with zipfile.ZipFile(z) as f:
        f.extractall(destino)
    print(f'  {nombre}: listo')
    return True

print('Descomprimiendo:')
descomprimir('paquete_colab.zip', BASE)
hay_sinteticas = descomprimir('sinteticas_variadas.zip', BASE / 'sinteticas')

## 2. Datos de entrenamiento

**AGAR.** Si tienes el dataset completo en Drive, indica aqui su carpeta. Si no,
se usa la muestra de 40 imagenes que viene en el paquete, suficiente para
comprobar que el flujo corre pero no para obtener un modelo util.

El conversor excluye las placas que AGAR marca con `colonies_number = -1`, que
son las que sus microbiologos consideraron incontables y vienen sin anotar, y
junta las cinco especies en una sola clase, porque para contar no hace falta
distinguirlas.

In [ ]:
ORIGEN_AGAR = BASE / 'agar_muestra'      # cambia por la carpeta de AGAR completo
AGAR_YOLO = BASE / 'agar_yolo'

proceso = subprocess.run(
    ['python', str(BASE / 'scripts' / 'agar_a_yolo.py'),
     str(ORIGEN_AGAR), '--salida', str(AGAR_YOLO)],
    capture_output=True, text=True)
print(proceso.stdout)
if proceso.returncode != 0:
    print('ERROR:', proceso.stderr)

**Conjunto combinado.** Se unen las imagenes de AGAR con las placas
sinteticas de captura variada. La idea es que AGAR aporte colonias reales y
abundantes, mientras que las sinteticas aporten la variabilidad de condiciones
que AGAR no tiene, porque sus fotografias proceden de un montaje profesional y
uniforme.

**Sobre las sinteticas.** Se generan pegando colonias reales, recortadas con las
coordenadas del conteo manual, sobre fondos de placa reales, y luego se simulan
distintas condiciones de captura. Su aspecto no es del todo convincente: las
colonias se repiten porque salen de un catalogo de 429 parches, el rango de
tamanos es estrecho, la distribucion espacial es uniforme cuando en una placa
real las colonias se agrupan, y no hay fusion ni colonias satelite. Por eso se
usan solo para entrenar y **nunca para evaluar**.

In [ ]:
COMBINADO = BASE / 'combinado'
for sub in ['train', 'val']:
    (COMBINADO / 'images' / sub).mkdir(parents=True, exist_ok=True)
    (COMBINADO / 'labels' / sub).mkdir(parents=True, exist_ok=True)

def copiar(origen, prefijo):
    total = 0
    for sub in ['train', 'val']:
        img_dir = origen / 'images' / sub
        if not img_dir.exists():
            continue
        for img in img_dir.glob('*.jpg'):
            lbl = origen / 'labels' / sub / f'{img.stem}.txt'
            if not lbl.exists():
                continue
            shutil.copy2(img, COMBINADO / 'images' / sub / f'{prefijo}_{img.name}')
            shutil.copy2(lbl, COMBINADO / 'labels' / sub / f'{prefijo}_{img.stem}.txt')
            total += 1
    return total

n_agar = copiar(AGAR_YOLO, 'agar')
print(f'AGAR: {n_agar} imagenes')

USAR_SINTETICAS = True     # ponlo en False para entrenar solo con AGAR
if USAR_SINTETICAS and hay_sinteticas:
    n_sint = copiar(BASE / 'sinteticas', 'sint')
    print(f'sinteticas: {n_sint} imagenes')

(COMBINADO / 'data.yaml').write_text(
    f'path: {COMBINADO.as_posix()}\n'
    'train: images/train\n'
    'val: images/val\n\n'
    'nc: 1\n'
    "names: ['colonia']\n")
print('\nConjunto combinado listo')

## 3. Entrenamiento

Se usa `imgsz=1280` porque las colonias son pequenas dentro de una placa grande
y reducir la imagen las hace desaparecer, que es el problema ya diagnosticado con
CellSAM.

La aumentacion se activa con generosidad en todo lo que tiene que ver con la
captura, es decir tono, saturacion, brillo y perspectiva, porque el objetivo es
justamente que el modelo no dependa de esas condiciones. Los volteos y la
rotacion libre se activan porque una placa no tiene orientacion privilegiada.

In [ ]:
modelo = YOLO('yolov8n.pt')

modelo.train(
    data=str(COMBINADO / 'data.yaml'),
    epochs=150,
    imgsz=1280,
    batch=8,
    patience=30,
    hsv_h=0.03,       # tono, generoso: distintos balances de blancos
    hsv_s=0.8,        # saturacion
    hsv_v=0.5,        # brillo, distintas exposiciones
    degrees=180,
    fliplr=0.5,
    flipud=0.5,
    scale=0.4,
    perspective=0.0005,
    mosaic=1.0,
    project='/content/runs',
    name='detector_robusto',
    exist_ok=True,
)

## 4. Evaluacion en dos montajes reales

Esta es la prueba que importa. **El mismo modelo, sin recalibrar nada**, se
aplica a fotografias de dos montajes distintos.

**Montaje A**, `images/placas`, iluminacion directa y dos placas por fotografia.
Como cada imagen contiene dos placas, se detectan y recortan antes de contar.

**Montaje B**, `images/mis_fotos`, transiluminador con contraluz y una placa por
fotografia.

Si el modelo rinde parecido en ambos, la hipotesis se sostiene y el sistema
podria usarse en un laboratorio nuevo sin ajustes. Si rinde bien en uno y mal en
otro, la variabilidad simulada no basto.

In [ ]:
import cv2
import numpy as np
import sys
sys.path.insert(0, str(BASE / 'scripts'))

MAX_DET = 500
CONF = 0.25

def contar(ruta, modelo):
    pred = modelo.predict(str(ruta), imgsz=1280, conf=CONF,
                          max_det=MAX_DET, verbose=False)[0]
    return len(pred.boxes)

# ── Montaje B, una placa por imagen ─────────────────────────────────────────
FOTOS = BASE / 'mis_fotos'
GT_B = pd.read_csv(BASE / 'ground_truth' / 'ground_truth.csv')
GT_B = GT_B.set_index('image')['count'].to_dict()

filas_b = []
for ruta in sorted(FOTOS.glob('*.jpg')):
    if ruta.stem.startswith(('M1', 'M2')):
        continue          # cultivos fallidos, el agar impidio el crecimiento
    n = contar(ruta, modelo)
    filas_b.append({'placa': ruta.stem, 'manual': GT_B.get(ruta.stem),
                    'sistema': n})

df_b = pd.DataFrame(filas_b)
df_b['error'] = df_b['sistema'] - df_b['manual']
df_b

In [ ]:
# ── Montaje A, dos placas por imagen ────────────────────────────────────────
# El recorte lo hace vision clasica: se parte la imagen por la mitad y se busca
# un circulo en cada mitad.
from contar_placas_dobles import detectar_dos_placas
from contar_mis_fotos import crop_plate

PLACAS = BASE / 'placas'      # incluye este directorio en el paquete si lo quieres evaluar
filas_a = []

if PLACAS.exists():
    gt_a = {}
    for _, f in pd.read_csv(PLACAS / 'ground_truth.csv').iterrows():
        gt_a[Path(f['image']).stem] = (int(f['plate_A']), int(f['plate_B']))

    for ruta in sorted(PLACAS.glob('*.jpeg')):
        img = cv2.imread(str(ruta))
        conteos = []
        for cx, cy, r in detectar_dos_placas(img):
            crop, _ = crop_plate(img, cx, cy, r)
            tmp = '/content/_tmp.jpg'
            cv2.imwrite(tmp, crop)
            conteos.append(contar(tmp, modelo))
        ma, mb = gt_a.get(ruta.stem, (None, None))
        filas_a.append({'imagen': ruta.stem, 'manual_A': ma, 'sistema_A': conteos[0],
                        'manual_B': mb, 'sistema_B': conteos[1]})
    df_a = pd.DataFrame(filas_a)
    display(df_a)
else:
    print('No se encontro el montaje A. Incluye images/placas en el paquete '
          'para evaluarlo.')

In [ ]:
# ── Metricas comparables entre montajes ─────────────────────────────────────
TNTC = 250

def resumen(manual, sistema, etiqueta):
    manual = pd.Series(manual).astype(float)
    sistema = pd.Series(sistema).astype(float)
    err = (sistema - manual).abs()
    cont = manual <= TNTC
    print(f'{etiqueta:<28} n={len(manual):>3}  '
          f'MAE_cont={err[cont].mean():>6.2f}  '
          f'MAE={err.mean():>6.2f}  '
          f'acierto={(1 - err.sum()/manual.sum())*100:>5.1f}%')

print('Un solo modelo, sin recalibrar entre montajes:\n')
resumen(df_b['manual'], df_b['sistema'], 'Montaje B, contraluz')
if filas_a:
    ma = list(df_a['manual_A']) + list(df_a['manual_B'])
    sa = list(df_a['sistema_A']) + list(df_a['sistema_B'])
    resumen(ma, sa, 'Montaje A, luz directa')

print('\nReferencia del pipeline con CellSAM, que se recalibra por montaje:')
print('  Montaje B  MAE 2.83, acierto 83.8%')
print('  Montaje A  MAE 5.94 con su calibracion propia')

## 5. Prueba ciega

Las cuatro placas NRC73 del segundo lote no participaron en ningun ajuste ni en
ninguna decision de parametros, y su conteo manual no se ha usado todavia. Son la
unica medida honesta de rendimiento sobre datos nuevos.

Esta celda produce los conteos. La comparacion con el conteo manual la hace la
investigadora, para preservar el caracter ciego de la prueba.

In [ ]:
LOTE2 = BASE / 'mis_fotos_lote2'
if LOTE2.exists():
    for ruta in sorted(LOTE2.glob('*.jpg')):
        print(f'{ruta.stem:18} {contar(ruta, modelo):>4} colonias')
else:
    print('No se encontro el segundo lote.')

## 6. Guardar el modelo

Colab borra el disco al cerrar la sesion, asi que conviene copiar los pesos a
Drive.

In [ ]:
destino = DRIVE / 'modelos_colonias'
destino.mkdir(exist_ok=True)
origen = Path('/content/runs/detector_robusto/weights/best.pt')
if origen.exists():
    shutil.copy2(origen, destino / 'detector_robusto.pt')
    print('guardado en', destino / 'detector_robusto.pt')

## Como leer los resultados

**Si el modelo rinde parecido en los dos montajes**, la hipotesis se sostiene: un
detector entrenado con suficiente variedad de condiciones no necesita
recalibrarse, y el sistema puede entregarse a otro laboratorio tal cual. Ese
seria el resultado que justifica el proyecto.

**Si rinde bien en uno y mal en otro**, la variabilidad simulada no basto. El
siguiente paso seria mejorar el generador de sinteticas por el orden ya
identificado: ampliar el catalogo de colonias, ampliar el rango de tamanos,
introducir agrupamiento espacial y acotar la variacion de color.

**Si rinde peor que el pipeline con CellSAM en ambos**, la conclusion tambien es
util y publicable: un detector entrenado especificamente no supera a un modelo de
fundacion adaptado, lo que refuerza la eleccion de diseno del trabajo.

En los tres casos conviene comparar contra las cifras del pipeline actual, que se
recalibra por montaje, para saber cuanto cuesta la robustez en terminos de
precision.